# Audit dan penguatan Tahap A — DSS Desa Wisata (1.336 identitas)

**Tujuan:** memperkuat klasifikasi kelompok *di luar benchmark* (`0`) vs *benchmark tingkat lanjut Maju/Mandiri* (`1`), **bukan** menetapkan status resmi JADESTA.

**Disiplin evaluasi**
- Menggunakan 987 + 315 + 34 identitas unik dari 3 CSV hasil pembersihan.
- Mempertahankan **holdout 80:20 yang sama** (`random_state=42`). **Tidak mengukur ulang, mengoptimalkan, atau memilih model berdasarkan hasil holdout lama**, karena holdout sebelumnya sudah dilihat. Untuk estimasi ketidakpastian digunakan *nested/repeated CV* **khusus 1.068 data training**.
- Bandingkan Logistic Regression, Calibrated Linear SVM, dan Random Forest dengan 4 studi ablasi. Semua TF-IDF/scaler dilatih **di dalam fold**.
- Model dan threshold dipilih pada *inner folds*, lalu diuji pada *outer folds* yang independen terhadap pemilihan tersebut.
- Dilakukan audit *missingness*, *sensitivity by regency* (geographical holdout), dan *paired bootstrap* tingkat **desa** tanpa memperlakukan pengulangan CV sebagai sampel independen.

**Batasan sangat penting:** 987 desa di luar benchmark belum terbukti semuanya bukan Maju/Mandiri; seluruh kolom `status_terverifikasi_JADESTA` masih `BELUM`. Eksperimen ini mengevaluasi **keanggotaan terhadap daftar benchmark sementara**, bukan prediksi kesiapan resmi atau efek kausal. Jika label terverifikasi berbeda, eksperimen wajib diulang.

## 0. Persiapan dan input tiga CSV

Jalankan di Google Colab. Jika berkas sudah ada di `/content`, tidak perlu unggah ulang. `DSS_FAST_MODE=1` **hanya untuk debugging**, bukan angka manuskrip. Secara default mode penelitian adalah penuh.

In [ ]:
import os, sys, re, json, zipfile, warnings, random, platform
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn import __version__ as sklearn_version
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report, brier_score_loss)
from scipy.stats import beta

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
FAST_MODE = os.environ.get('DSS_FAST_MODE', '0') == '1'  # Default: MODE PENELITIAN
CV_FOLDS = 3 if FAST_MODE else 5
RF_TREES = 55 if FAST_MODE else 160
SHAP_BG_PER_CLASS = (3, 3, 3) if FAST_MODE else (10, 6, 4)
SHAP_EXPLAIN_PER_CLASS = (3, 2, 1) if FAST_MODE else (12, 9, 6)
SHAP_N_PERMUTATIONS = 1 if FAST_MODE else 2

OUT_DIR = Path('/content/hasil_robustness_tahap_A' if Path('/content').exists() else 'hasil_robustness_tahap_A')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = {
    0: 'DESA_DI_LUAR_BENCHMARK_987.csv',
    1: 'DESA_MAJU_315.csv',
    2: 'DESA_MANDIRI_34.csv',
}
LABEL_NAMES = {0: 'Di Luar Benchmark', 1: 'Maju', 2: 'Mandiri'}
print('FAST_MODE:', FAST_MODE, '| CV_FOLDS:', CV_FOLDS, '| output:', OUT_DIR.resolve())
print('scikit-learn:', sklearn_version)

FAST_MODE: False | CV_FOLDS: 5 | output: /content/hasil_robustness_tahap_A
scikit-learn: 1.6.1


In [ ]:
import time, itertools, joblib, hashlib
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.dummy import DummyClassifier

FAST_MODE = os.environ.get('DSS_FAST_MODE','0') == '1'
SCREEN_REPEATS = 1 if FAST_MODE else 3
OUTER_REPEATS = 1 if FAST_MODE else 2
OUTER_FOLDS = 3 if FAST_MODE else 5
INNER_FOLDS = 2 if FAST_MODE else 3
RF_TREES_A = 35 if FAST_MODE else 140
BOOTSTRAP_N = 150 if FAST_MODE else 1500
# PILIHAN: seluruh 12 kandidat pada CV nested. Jangan mengubah berdasarkan hasil holdout.
print('Konfigurasi:', {'FAST_MODE':FAST_MODE,'screen_repeats':SCREEN_REPEATS,'outer':(OUTER_FOLDS,OUTER_REPEATS),'inner':INNER_FOLDS,'bootstrap':BOOTSTRAP_N})

Konfigurasi: {'FAST_MODE': False, 'screen_repeats': 3, 'outer': (5, 2), 'inner': 3, 'bootstrap': 1500}


## 1. Pembacaan dan validasi data (tanpa merancang ulang label)

Normalisasi fitur mengikuti notebook DSS versi sebelumnya, supaya perbandingan tetap adil. Identitas dan sumber label **tidak dipakai sebagai prediktor**.

In [ ]:
missing_files = [name for name in FILES.values() if not Path(name).exists()]
if missing_files:
    try:
        from google.colab import files as colab_files
        print('Silakan unggah semua CSV berikut:', missing_files)
        colab_files.upload()
    except ImportError as exc:
        raise FileNotFoundError('Salin file berikut ke folder kerja: ' + ', '.join(missing_files)) from exc

frames = []
for label, filename in FILES.items():
    if not Path(filename).exists():
        raise FileNotFoundError('CSV tidak tersedia: ' + filename)
    part = pd.read_csv(filename, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    part['label_3kelas'] = label
    part['status_benchmark'] = LABEL_NAMES[label]
    part['file_sumber_label'] = filename
    frames.append(part)
    print(filename, '->', len(part), 'baris')

df_raw = pd.concat(frames, ignore_index=True)
assert df_raw.shape[0] == 1336, f'Jumlah data berubah: {len(df_raw)}. Audit sumber sebelum melanjutkan.'
assert df_raw['label_3kelas'].value_counts().to_dict() == {0: 987, 1: 315, 2: 34}
required_columns = ['Desa Wisata','Kabupaten','Deskripsi','Fasilitas','Atraksi','homestay',
                    'pakert wisata','sovenir','HP','IG','FB','Email','Website','Twitter','matching_key']
missing_cols = sorted(set(required_columns) - set(df_raw.columns))
assert not missing_cols, f'Kolom hilang: {missing_cols}'
assert (df_raw['matching_key'].astype(str).str.strip() != '').all(), 'Ada matching_key kosong'
assert not df_raw['matching_key'].duplicated().any(), 'Identitas desa terduplikasi. Periksa sumber!'
assert (df_raw['status_terverifikasi_JADESTA'].str.upper() == 'BELUM').all(), 'Status verifikasi berubah; periksa ulang metodologi.'

summary = df_raw.groupby(['label_3kelas','status_benchmark']).size().reset_index(name='jumlah')
summary['persentase'] = (summary['jumlah'] / len(df_raw) * 100).round(2)
display(summary)
summary.to_csv(OUT_DIR/'01_distribusi_label.csv', index=False, encoding='utf-8-sig')
print('Total identitas unik:', df_raw['matching_key'].nunique())
print('PENTING: Seluruh label adalah benchmark sementara, bukan verifikasi JADESTA.')

Silakan unggah semua CSV berikut: ['DESA_DI_LUAR_BENCHMARK_987.csv', 'DESA_MAJU_315.csv', 'DESA_MANDIRI_34.csv']


Saving DESA_DI_LUAR_BENCHMARK_987.csv to DESA_DI_LUAR_BENCHMARK_987.csv
Saving DESA_MAJU_315.csv to DESA_MAJU_315.csv
Saving DESA_MANDIRI_34.csv to DESA_MANDIRI_34.csv
DESA_DI_LUAR_BENCHMARK_987.csv -> 987 baris
DESA_MAJU_315.csv -> 315 baris
DESA_MANDIRI_34.csv -> 34 baris


,label_3kelas,status_benchmark,jumlah,persentase
0,0,Di Luar Benchmark,987,73.88
1,1,Maju,315,23.58
2,2,Mandiri,34,2.54


Total identitas unik: 1336
PENTING: Seluruh label adalah benchmark sementara, bukan verifikasi JADESTA.


## 2. Rekonstruksi 12 fitur terstruktur dan blok TF-IDF

- Produk/fasilitas/akomodasi: enam fitur numerik.
- Kanal digital: enam fitur, termasuk telepon dan email.
- Seluruh teks destinasi digabung sebagai satu kolom mentah untuk TF-IDF; nama desa/kabupaten tidak dimasukkan.
- Kolom `status_benchmark`/`matching_key`/`label_3kelas` hanya dipakai untuk pencocokan/evaluasi, **tidak masuk fitur**.

In [ ]:
EMPTY_VALUES = {'', '-', '--', '---', 'nan', 'none', 'null', 'n/a', 'na', 'tidak ada', 'belum ada', 'tdk ada'}
PSEUDO = {
    'Fasilitas': {'fasilitas', 'belum ada fasilitas untuk ditampilkan', 'belum tersedia'},
    'Atraksi': {'atraksi wisata', 'atraksi', 'belum ada atraksi untuk ditampilkan'},
    'homestay': {'kamar homestay', 'homestay', 'belum ada homestay untuk ditampilkan'},
    'pakert wisata': {'paket wisata', 'belum ada paket wisata untuk ditampilkan'},
    'sovenir': {'suvenir', 'souvenir', 'belum ada suvenir untuk ditampilkan'},
}

def clean_text(value):
    if pd.isna(value):
        return ''
    text = str(value).strip()
    if text.lower() in EMPTY_VALUES:
        return ''
    # Hapus ekspresi penghargaan/ranking yang berpotensi membocorkan label
    text = re.sub(r'\b\d+\s*besar\s*adwi\s*(?:20\d{2})?\b', ' ', text, flags=re.I)
    text = re.sub(r'\badwi\s*(?:20\d{2})?\b', ' ', text, flags=re.I)
    text = re.sub(r'\b(?:status|klasifikasi|kategori)\s+(?:desa\s+wisata\s+)?(?:mandiri|maju|rintisan|berkembang)\b', ' ', text, flags=re.I)
    return re.sub(r'\s+', ' ', text).strip()

def count_documented_items(value, column):
    text = clean_text(value)
    if not text or 'belum ada' in text.lower() or 'belum tersedia' in text.lower():
        return 0
    # Harga dalam format Rp 150,000 tidak boleh dihitung sebagai 2 entri
    text = re.sub(r'(?<=\d),(?=\d{3}(?:\D|$))', '', text)
    parts = re.split(r'[,;\n\r|]+', text)
    items = []
    for item in parts:
        item = item.strip(' .-_')
        if not item or item.lower() in PSEUDO[column]:
            continue
        for heading in PSEUDO[column]:
            if item.lower().startswith(heading + ' ') and len(item) > len(heading) + 2:
                item = item[len(heading):].strip()
        if item and item.lower() not in PSEUDO[column]:
            items.append(item)
    return len(items)

def valid_channel(value, channel):
    s = clean_text(value)
    if not s:
        return 0
    if channel == 'HP':
        return int(len(re.sub(r'\D', '', s)) >= 8)
    if channel == 'Email':
        return int(bool(re.search(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', s)))
    url = s if re.match(r'^https?://', s, flags=re.I) else 'https://' + s
    parsed = urlparse(url)
    host = parsed.netloc.lower().removeprefix('www.')
    slug = parsed.path.strip('/').lower()
    if channel == 'IG':
        return int(host in ('instagram.com','instagr.am') and slug not in ('','p','reel','stories','explore','jadestaofficial'))
    if channel == 'FB':
        return int(host in ('facebook.com','m.facebook.com','fb.com') and slug not in ('','jadestaofficial'))
    if channel == 'Twitter':
        return int(host in ('twitter.com','x.com') and slug not in ('','jadestaofficial','home','search'))
    if channel == 'Website':
        banned = ('facebook.com','instagram.com','twitter.com','x.com','wa.me','whatsapp.com',
                  'wa.link','linktr.ee','jadesta.kemenpar.go.id','jadesta.kemenparekraf.go.id')
        return int(bool(host and '.' in host and not any(host == d or host.endswith('.'+d) for d in banned)))
    return 0

NUM_COLS = ['description_length','num_facilities','num_attractions','num_homestay',
            'num_tour_packages','num_souvenirs','has_phone','has_instagram',
            'has_facebook','has_email','has_website','has_twitter']
TEXT_COL = 'combined_text_model'
TEXT_SOURCE_COLS = ['Deskripsi','Fasilitas','Atraksi','homestay','pakert wisata','sovenir']

df = df_raw.copy()
for col in required_columns:
    df[col] = df[col].fillna('').astype(str)
for col in TEXT_SOURCE_COLS:
    df['clean_' + col] = df[col].map(clean_text)

df['description_length'] = df['clean_Deskripsi'].map(lambda x: len(re.findall(r'\b\w+\b', x)))
for col, feature in [('Fasilitas','num_facilities'),('Atraksi','num_attractions'),
                     ('homestay','num_homestay'),('pakert wisata','num_tour_packages'),('sovenir','num_souvenirs')]:
    df[feature] = df[col].map(lambda x: count_documented_items(x, col))
for col, feature in [('HP','has_phone'),('IG','has_instagram'),('FB','has_facebook'),
                     ('Email','has_email'),('Website','has_website'),('Twitter','has_twitter')]:
    df[feature] = df[col].map(lambda x: valid_channel(x, col))

df[TEXT_COL] = df[['clean_' + col for col in TEXT_SOURCE_COLS]].agg(' '.join,axis=1).str.replace(r'\s+',' ',regex=True).str.strip()
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors='raise').astype(float)
df['digital_presence_score'] = df[['has_instagram','has_facebook','has_email','has_website','has_twitter']].sum(axis=1)
df['product_readiness_score'] = df[['num_attractions','num_tour_packages','num_souvenirs']].sum(axis=1)
df['infrastructure_readiness_score'] = df[['num_facilities','num_homestay']].sum(axis=1)
assert df[NUM_COLS+[TEXT_COL]].notna().all().all()
assert (df[NUM_COLS].to_numpy() >= 0).all()

X_ALL = df[NUM_COLS+[TEXT_COL]].copy()
y_all = df['label_3kelas'].astype(int)
print('Matriks mentah prediktor:',X_ALL.shape,'| 12 numerik + 1 blok teks')
display(df.groupby('status_benchmark')[NUM_COLS].agg(['mean','median']).round(2).iloc[:,:12])
df[['matching_key','label_3kelas']+NUM_COLS+['digital_presence_score','product_readiness_score','infrastructure_readiness_score']].to_csv(
    OUT_DIR/'02_fitur_terstruktur_audit.csv',index=False,encoding='utf-8-sig')

Matriks mentah prediktor: (1336, 13) | 12 numerik + 1 blok teks


description_length        num_facilities         \
                                mean median           mean median   
status_benchmark                                                    
Di Luar Benchmark             186.45   89.0           6.57    7.0   
Maju                          215.37  135.0           8.82   10.0   
Mandiri                        65.68   51.0           9.71   10.0   

                  num_attractions        num_homestay         \
                             mean median         mean median   
status_benchmark                                               
Di Luar Benchmark            4.25    2.0         1.51    0.0   
Maju                         6.91    5.0         3.38    2.0   
Mandiri                      6.24    4.0         0.65    0.0   

                  num_tour_packages        num_souvenirs         
                               mean median          mean median  
status_benchmark                                                 
Di Luar Benchmark              2.21    1.0          1.76    0.0  
Maju                           3.77    2.0          2.78    2.0  
Mandiri                        3.71    1.0          1.97    0.5

## 3. Pembekuan split holdout dan audit identitas

Jika file ZIP hasil lama tersedia, bandingkan `matching_key` dari pembagian train/test. Notebook ini **tidak melatih/mengevaluasi pada holdout**; hasil 20% hanya dipakai untuk mengaudit konsistensi pembagian.

In [ ]:
train_idx, test_idx = train_test_split(df.index.to_numpy(), test_size=0.20, stratify=y_all, random_state=SEED)
train_idx, test_idx = np.sort(train_idx), np.sort(test_idx)
assert len(set(train_idx) & set(test_idx)) == 0
train = df.loc[train_idx].copy()
test = df.loc[test_idx].copy()
X_train = train[NUM_COLS + [TEXT_COL]].copy()
X_test = test[NUM_COLS + [TEXT_COL]].copy()
y_A_train = (train['label_3kelas'] >= 1).astype(int)
y_A_test = (test['label_3kelas'] >= 1).astype(int)
train_B = train[train['label_3kelas'].isin([1,2])].copy()
test_B = test[test['label_3kelas'].isin([1,2])].copy()
X_B_train = train_B[NUM_COLS + [TEXT_COL]].copy()
X_B_test = test_B[NUM_COLS + [TEXT_COL]].copy()
y_B_train = (train_B['label_3kelas'] == 2).astype(int)
y_B_test = (test_B['label_3kelas'] == 2).astype(int)

split_table = pd.concat([
    train['label_3kelas'].value_counts().sort_index().rename('train_80'),
    test['label_3kelas'].value_counts().sort_index().rename('test_20')], axis=1).fillna(0).astype(int)
split_table['total'] = split_table.sum(axis=1)
split_table['kelas'] = split_table.index.map(LABEL_NAMES)
display(split_table)
split_table.to_csv(OUT_DIR/'03_split_train_test.csv',encoding='utf-8-sig')
train[['matching_key','Desa Wisata','Kabupaten','label_3kelas']].to_csv(OUT_DIR/'03_train_ids.csv',index=False,encoding='utf-8-sig')
test[['matching_key','Desa Wisata','Kabupaten','label_3kelas']].to_csv(OUT_DIR/'03_test_ids.csv',index=False,encoding='utf-8-sig')

,train_80,test_20,total,kelas
label_3kelas,,,,
0,789,198,987,Di Luar Benchmark
1,252,63,315,Maju
2,27,7,34,Mandiri


In [ ]:
# Jika ZIP hasil eksperimen sebelumnya diunggah, pastikan pembagian tidak berubah.
prior_zip = Path('HASIL_DSS_1336_SHAP.zip')
if prior_zip.exists():
    with zipfile.ZipFile(prior_zip) as z:
        old_train=pd.read_csv(z.open('03_train_ids.csv'),dtype=str)['matching_key'].tolist()
        old_test=pd.read_csv(z.open('03_test_ids.csv'),dtype=str)['matching_key'].tolist()
    assert set(old_train)==set(train['matching_key']), 'Split TRAIN berbeda dengan eksperimen lama.'
    assert set(old_test)==set(test['matching_key']), 'Split TEST berbeda dengan eksperimen lama.'
    print('PASS: train/test identik dengan eksperimen DSS sebelumnya.')
else:
    print('ZIP lama opsional tidak tersedia. Split dihitung ulang deterministik dengan seed 42.')
assert len(train)==1068 and len(test)==268
assert not set(train['matching_key']) & set(test['matching_key'])
assert train['matching_key'].is_unique and test['matching_key'].is_unique
assert int(y_A_train.sum())==279 and int(y_A_test.sum())==70
print('TRAIN',len(train),'(positif proxy',int(y_A_train.sum()),') | HOLDOUT TERKUNCI',len(test),'(positif proxy',int(y_A_test.sum()),')')

PASS: train/test identik dengan eksperimen DSS sebelumnya.
TRAIN 1068 (positif proxy 279 ) | HOLDOUT TERKUNCI 268 (positif proxy 70 )


## 4. Audit kualitas label dan data hilang

Audit difokuskan pada perbedaan proporsi pengisian informasi antar kelompok. Pola hilang yang berbeda dapat membuat model mempelajari **kebiasaan dokumentasi**, bukan kematangan destinasi.

In [ ]:
quality=[]
for target_col,flag_col in [('Deskripsi','description_length'),('Website','has_website'),('IG','has_instagram'),('FB','has_facebook'),('Twitter','has_twitter')]:
    for label,g in train.groupby('label_3kelas'):
        q={'kolom':target_col,'kelas':LABEL_NAMES[int(label)],'n':len(g)}
        if target_col=='Deskripsi':q['persen_tidak_terdokumentasi']=100*(g['description_length']==0).mean()
        else:q['persen_tidak_terdokumentasi']=100*(g[flag_col]==0).mean()
        quality.append(q)
quality_df=pd.DataFrame(quality)
quality_df.to_csv(OUT_DIR/'01_audit_kelengkapan_train.csv',index=False)
display(quality_df.pivot(index='kolom',columns='kelas',values='persen_tidak_terdokumentasi').round(1))

# Periksa apakah string status eksplisit masih tersisa pada fitur teks
status_pattern = r'(?i)\b(?:kategori|klasifikasi|berstatus|status)\s+(?:desa\s+wisata\s+)?(?:mandiri|maju|berkembang|rintisan)\b'
leaks=train[train[TEXT_COL].str.contains(status_pattern,regex=True,na=False)]
print('Sisa pola status eksplisit di teks TRAIN:',len(leaks))
leaks[['matching_key','label_3kelas']].to_csv(OUT_DIR/'01_audit_kata_status.csv',index=False)

## 5. Studi ablasi terdaftar dan tiga algoritma

Empat set fitur: **structured only**, **TF-IDF only**, **structured+TF-IDF**, dan **structured tanpa kanal digital + TF-IDF**. Seluruh pra-pemrosesan dipasang di dalam `Pipeline` agar tidak terjadi *vocabulary/data leakage*. Dalam model berbasis RF, seluruh TF-IDF juga tetap berada di pipeline.

Evaluasi memilih model berdasarkan **PR-AUC (average precision)**, sedangkan F1/BA dipakai sebagai pendamping; tidak memilih berdasarkan akurasi total.

In [ ]:
NUM_CORE = ['description_length','num_facilities','num_attractions','num_homestay','num_tour_packages','num_souvenirs']
ABLATIONS = {
 'structured_only': {'num':NUM_COLS, 'text':False},
 'tfidf_only': {'num':[], 'text':True},
 'structured_plus_tfidf': {'num':NUM_COLS, 'text':True},
 'non_digital_plus_tfidf': {'num':NUM_CORE, 'text':True},
}
MODELS_A=['Logistic Regression','Calibrated Linear SVM','Random Forest']
CANDIDATES=[(m,a) for m in MODELS_A for a in ABLATIONS]

def stageA_estimator(model_name, ablation_name):
    cfg=ABLATIONS[ablation_name]
    transformers=[]
    if cfg['num']:
        transformers.append(('numerik',StandardScaler(),cfg['num']))
    if cfg['text']:
        transformers.append(('tfidf',TfidfVectorizer(
            ngram_range=(1,2), min_df=2,max_df=0.95,max_features=3000,
            sublinear_tf=True,strip_accents=None),TEXT_COL))
    feats=ColumnTransformer(transformers,remainder='drop',sparse_threshold=0.30)
    if model_name=='Logistic Regression':
        return Pipeline([('fitur',feats),('model',LogisticRegression(
            solver='liblinear',class_weight='balanced',max_iter=2500,random_state=42))])
    if model_name=='Calibrated Linear SVM':
        base=Pipeline([('fitur',feats),('model',LinearSVC(
            dual='auto',max_iter=6000,class_weight='balanced',random_state=42))])
        return CalibratedClassifierCV(estimator=base,method='sigmoid',cv=3)
    if model_name=='Random Forest':
        return Pipeline([('fitur',feats),('model',RandomForestClassifier(
            n_estimators=RF_TREES_A,min_samples_leaf=2,class_weight='balanced_subsample',
            n_jobs=-1,random_state=42))])
    raise KeyError(model_name)

TRAIN_INPUTS=NUM_COLS+[TEXT_COL]
Xa=train[TRAIN_INPUTS].reset_index(drop=True)
ya=(train['label_3kelas'].to_numpy()>=1).astype(int)
assert len(Xa)==len(ya)==1068
assert len(CANDIDATES)==12

## 6. Repeated Stratified 5-Fold CV — perbandingan eksploratif

Hasil `repeated CV` bermanfaat melihat variasi pembagian data. **Fold dari pengulangan berbeda bukan observasi independen**, sehingga nilai simpangan bakunya tidak boleh langsung dijadikan *standard error* atau dasar uji-t antar-fold. Analisis berikutnya menggunakan nested CV untuk estimasi yang tidak memakai fold seleksi sebagai fold evaluasi.

In [ ]:
score_names={'ap':'average_precision','auc':'roc_auc','balacc':'balanced_accuracy',
             'f1':'f1','recall':'recall','precision':'precision','accuracy':'accuracy'}
screener=RepeatedStratifiedKFold(n_splits=OUTER_FOLDS,n_repeats=SCREEN_REPEATS,random_state=SEED)
screen_rows=[]
start=time.time()
for model_name,ablation_name in CANDIDATES:
    print(f'Screen {model_name:24} | {ablation_name:26}',flush=True)
    sc=cross_validate(stageA_estimator(model_name,ablation_name), Xa,ya,
                      cv=screener,scoring=score_names,n_jobs=1,error_score='raise')
    row={'model':model_name,'ablation':ablation_name,'n_folds':len(sc['test_ap'])}
    for met in score_names:
        a=sc['test_'+met]
        row[met+'_mean']=float(a.mean());row[met+'_sd_fold_deskriptif']=float(a.std(ddof=1))
    screen_rows.append(row)
screen=pd.DataFrame(screen_rows).sort_values(['ap_mean','balacc_mean'],ascending=False).reset_index(drop=True)
screen.to_csv(OUT_DIR/'02_screening_12_kombinasi.csv',index=False)
print(f'Durasi screening: {time.time()-start:.1f} detik (hanya debugging jika FAST_MODE).')
display(screen[['model','ablation','ap_mean','ap_sd_fold_deskriptif','balacc_mean','f1_mean','recall_mean']].round(4))

## 7. Nested CV — pemilihan model+ablasi+threshold di dalam data pelatihan

Untuk **setiap outer fold**: (1) pada *inner training* bandingkan seluruh 12 kandidat menurut mean AP; (2) pilih satu kandidat; (3) lakukan prediksi OOF inner untuk optimasi threshold F1; (4) latih kandidat pada seluruh outer training; (5) prediksi outer validation. Semua tahap seleksi **tidak menyentuh outer validation**.

Pengulangan outer CV mengevaluasi sensitivitas terhadap pemisahan data. Data holdout lama tetap tidak disentuh.

In [ ]:
def choose_threshold(y,p):
    grid=[]
    for th in np.round(np.arange(0.10,0.901,0.02),2):
        pr=(np.asarray(p)>=th).astype(int)
        grid.append((float(th),float(f1_score(y,pr,zero_division=0)),
                     float(balanced_accuracy_score(y,pr)),float(recall_score(y,pr,zero_division=0))))
    # Urutan deterministik: F1, BA, recall, paling dekat dari 0.5
    return max(grid,key=lambda t:(t[1],t[2],t[3],-abs(t[0]-0.5)))[0]

def metrics_row(y,p,pred):
    return {'pr_auc':float(average_precision_score(y,p)),
            'roc_auc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
            'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'f1':float(f1_score(y,pred,zero_division=0)),
            'precision':float(precision_score(y,pred,zero_division=0)),
            'recall':float(recall_score(y,pred,zero_division=0)),
            'accuracy':float(accuracy_score(y,pred)),
            'brier':float(brier_score_loss(y,p))}

# Default full: outer 5 folds x 2 repeats; inner 3 folds. FAST: outer 3x1, inner 2.
all_records=[]; fold_records=[]; starts=time.time()
for rep in range(OUTER_REPEATS):
    outer=StratifiedKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=SEED+rep)
    for fold,(idx_fit,idx_eval) in enumerate(outer.split(Xa,ya),start=1):
        Xin,yin=Xa.iloc[idx_fit],ya[idx_fit]
        Xout,yout=Xa.iloc[idx_eval],ya[idx_eval]
        icv=StratifiedKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=SEED+300+fold+rep*100)
        candidates_rank=[]
        for model_name,ablation_name in CANDIDATES:
            sc=cross_validate(stageA_estimator(model_name,ablation_name),Xin,yin,
                    cv=icv,scoring={'ap':'average_precision','ba':'balanced_accuracy'},
                    n_jobs=1,error_score='raise')
            candidates_rank.append((float(sc['test_ap'].mean()),float(sc['test_ba'].mean()),model_name,ablation_name))
        # Tie break deterministik berdasarkan nama
        best=sorted(candidates_rank,key=lambda x:(-x[0],-x[1],x[2],x[3]))[0]
        _,_,best_name,best_ablation=best
        chosen=stageA_estimator(best_name,best_ablation)
        p_oof=cross_val_predict(chosen,Xin,yin,cv=icv,method='predict_proba',n_jobs=1)[:,1]
        chosen_threshold=choose_threshold(yin,p_oof)
        chosen.fit(Xin,yin)
        p=chosen.predict_proba(Xout)[:,1]
        pred=(p>=chosen_threshold).astype(int)

        # Pembanding pra-spesifikasi: Logistic Regression + 12 numerik + TF-IDF,
        # threshold dipilih via OOF inner dengan data outer-training saja.
        baseline=stageA_estimator('Logistic Regression','structured_plus_tfidf')
        p_baseline_oof=cross_val_predict(baseline,Xin,yin,cv=icv,method='predict_proba',n_jobs=1)[:,1]
        th_baseline=choose_threshold(yin,p_baseline_oof)
        baseline.fit(Xin,yin)
        pb=baseline.predict_proba(Xout)[:,1]
        predb=(pb>=th_baseline).astype(int)
        fm=metrics_row(yout,p,pred)
        fb=metrics_row(yout,pb,predb)
        fold_records.append({'repeat':rep+1,'fold':fold,'selected_model':best_name,
             'selected_ablation':best_ablation,'inner_mean_ap':best[0],
             'threshold_selected':chosen_threshold,'threshold_baseline':th_baseline,
             'n_outer_test':len(idx_eval),'outer_pos':int(yout.sum()),
             **{'selected_'+k:v for k,v in fm.items()},
             **{'baseline_'+k:v for k,v in fb.items()}})
        for j,original_pos in enumerate(idx_eval):
            all_records.append({'repeat':rep+1,'fold':fold,
                'matching_key':train.iloc[original_pos]['matching_key'],
                'y':int(yout[j]),'p_selected':float(p[j]),'pred_selected':int(pred[j]),
                'p_legacy_lr':float(pb[j]),'pred_legacy_lr':int(predb[j]),
                'threshold_selected':chosen_threshold,'threshold_legacy_lr':th_baseline,
                'selected_model':best_name,'selected_ablation':best_ablation})
        print(f'Nested repeat {rep+1}/{OUTER_REPEATS}, fold {fold}/{OUTER_FOLDS}: '
              f'{best_name} + {best_ablation}, AP={fm["pr_auc"]:.3f}, F1={fm["f1"]:.3f}',flush=True)
print('Durasi nested:',round(time.time()-starts,1),'detik')
cv_pred=pd.DataFrame(all_records)
fold_df=pd.DataFrame(fold_records)
assert len(cv_pred)==1068*OUTER_REPEATS
assert cv_pred.groupby(['repeat','matching_key']).size().eq(1).all()
cv_pred.to_csv(OUT_DIR/'03_nested_oof_predictions.csv',index=False)
fold_df.to_csv(OUT_DIR/'03_nested_folds.csv',index=False)

## 8. Ringkasan performa nested dan perbandingan baseline mayoritas

Gunakan hasil **OOF tingkat desa pada satu pengulangan penuh**, bukan rerata skor fold yang dianggap independen. Prevalensi kelas positif dihitung hanya pada subset train.

In [ ]:
repeat_rows=[]
for rep,g in cv_pred.groupby('repeat'):
    y=g['y'].to_numpy()
    assert len(g)==len(Xa) and g['matching_key'].nunique()==len(Xa)
    for name,pcol,dcol in [('nested_selected','p_selected','pred_selected'),
                            ('legacy_lr','p_legacy_lr','pred_legacy_lr')]:
        row={'repeat':int(rep),'model':name}
        row.update(metrics_row(y,g[pcol].to_numpy(),g[dcol].to_numpy()))
        repeat_rows.append(row)
    # Skenario naif selalu prediksi 0; AP dari probabilitas konstan = prevalensi
    repeat_rows.append({'repeat':int(rep),'model':'all_negative',
        **metrics_row(y,np.full(len(y),np.mean(y)),np.zeros(len(y),dtype=int))})
summary=pd.DataFrame(repeat_rows)
summary.to_csv(OUT_DIR/'04_nested_metrics_per_repeat.csv',index=False)
display(summary.round(4))
print('PERINGATAN: rerata antar repeat = deskriptif sensitivitas, bukan n sampel tambahan.')

## 9. Paired stratified bootstrap 95% CI — satu pengulangan OOF

*Bootstrap* tingkat desa pada prediksi OOF repeat pertama; pasangan model dipertahankan pada sampel desa yang sama. Ini adalah interval pendekatan kondisional terhadap prediksi OOF tersebut, **bukan** interval penuh yang memasukkan seluruh ketidakpastian pemilihan model/label. Jangan melaporkan uji-t atas fold repeated CV.

In [ ]:
rep1=cv_pred[cv_pred['repeat']==1].reset_index(drop=True)
y=rep1['y'].to_numpy(); ps=rep1['p_selected'].to_numpy();pb=rep1['p_legacy_lr'].to_numpy()
ys=rep1['pred_selected'].to_numpy();yb=rep1['pred_legacy_lr'].to_numpy()
neg=np.flatnonzero(y==0);pos=np.flatnonzero(y==1)
rng=np.random.default_rng(SEED+567)
boots=[]
for _ in range(BOOTSTRAP_N):
    indices=np.concatenate([rng.choice(neg,len(neg),replace=True),rng.choice(pos,len(pos),replace=True)])
    yy=y[indices]
    bs=metrics_row(yy,ps[indices],ys[indices]);bb=metrics_row(yy,pb[indices],yb[indices])
    boots.append({'selected_AP':bs['pr_auc'],'selected_F1':bs['f1'],'selected_BA':bs['balanced_accuracy'],
                  'delta_AP':bs['pr_auc']-bb['pr_auc'],
                  'delta_F1':bs['f1']-bb['f1'],
                  'delta_BA':bs['balanced_accuracy']-bb['balanced_accuracy']})
boots=pd.DataFrame(boots)
ci=boots.quantile([0.025,0.5,0.975]).T
ci.columns=['CI_2_5%','median','CI_97_5%']
ci.to_csv(OUT_DIR/'05_bootstrap_95CI.csv')
display(ci.round(4))
print('Peringatan: label benchmark belum terverifikasi; 95% CI bersifat exploratif.')

## 10. Sensitivitas terhadap kabupaten/kota yang belum terlihat

`StratifiedGroupKFold` memastikan seluruh kabupaten/kota pada fold evaluasi tidak muncul pada fold pelatihan. Ini uji **transfer geografis internal**, bukan validasi eksternal nasional. Model+ablasi untuk uji ini ditetapkan menurut frekuensi kemenangan nested CV, bukan berdasarkan hasil *group holdout*. Threshold dihitung dari OOF wilayah pelatihan saja.

In [ ]:
select_counts=fold_df.groupby(['selected_model','selected_ablation']).size().sort_values(ascending=False)
# Resolusi tie urutan leksikal stabil
winner_key=sorted(select_counts.items(),key=lambda kv:(-kv[1],kv[0][0],kv[0][1]))[0][0]
print('Kandidat paling sering terpilih nested:',winner_key,'dengan',int(select_counts.loc[winner_key]),'dari',len(fold_df),'outer folds')
groups=train['kabupaten_model'].fillna('').astype(str).replace('','kabupaten_tidak_tercatat').reset_index(drop=True)
print('Kelompok geografis unik:',groups.nunique())
cv_geo=StratifiedGroupKFold(n_splits=5 if not FAST_MODE else 3,shuffle=True,random_state=SEED)
geo_rows=[]
for i,(a,b) in enumerate(cv_geo.split(Xa,ya,groups=groups),start=1):
    assert not set(groups.iloc[a]) & set(groups.iloc[b])
    yin=ya[a]; yout=ya[b]
    if min(np.bincount(yin))<INNER_FOLDS: raise ValueError('Data positif terlalu sedikit pada fold geo training.')
    innergeo=StratifiedKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=SEED+10+i)
    est=stageA_estimator(*winner_key)
    oof=cross_val_predict(est,Xa.iloc[a],yin,cv=innergeo,method='predict_proba',n_jobs=1)[:,1]
    th=choose_threshold(yin,oof)
    est.fit(Xa.iloc[a],yin)
    p=est.predict_proba(Xa.iloc[b])[:,1]
    pred=(p>=th).astype(int)
    rr={'fold':i,'n':len(b),'n_pos':int(yout.sum()),'kabupaten_uji':int(groups.iloc[b].nunique()),'threshold':th}
    # Per-class metrics untuk fold dengan kelas tunggal tidak dianggap sebanding.
    if len(np.unique(yout))<2:
        rr.update({'pr_auc':np.nan,'roc_auc':np.nan,'balanced_accuracy':np.nan,
                   'f1':np.nan,'precision':np.nan,'recall':np.nan,'accuracy':float(accuracy_score(yout,pred)),'brier':float(brier_score_loss(yout,p))})
    else:rr.update(metrics_row(yout,p,pred))
    geo_rows.append(rr)
geo_df=pd.DataFrame(geo_rows)
geo_df.to_csv(OUT_DIR/'06_group_holdout_kabupaten.csv',index=False)
display(geo_df.round(4))

## 11. Grafik PR-AUC dan ringkasan pemenang model

Grafik menggunakan prediksi nested OOF repeat pertama pada data train, bukan holdout. Tabel seleksi menunjukkan apakah ada model+fitur yang menang stabil atau hasil sangat bergantung pada split.

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
PrecisionRecallDisplay.from_predictions(y,ps,name='Nested selected',ax=ax)
PrecisionRecallDisplay.from_predictions(y,pb,name='Legacy LR+combined',ax=ax)
ax.axhline(y.mean(),linestyle='--',linewidth=1,label=f'Prevalensi positif ({y.mean():.3f})')
ax.legend();ax.set_title('Tahap A — Precision-Recall (nested OOF, training only)')
fig.tight_layout();fig.savefig(OUT_DIR/'07_precision_recall_nested.png',dpi=200,bbox_inches='tight');plt.show()
selection=(fold_df.groupby(['selected_model','selected_ablation']).size()
             .reset_index(name='jumlah_outer_fold').sort_values('jumlah_outer_fold',ascending=False))
selection.to_csv(OUT_DIR/'07_frekuensi_pemilihan_kandidat.csv',index=False)
display(selection)

## 12. Ekspor model Tahap A yang dipilih TANPA evaluasi ulang pada holdout

Pilihan *deployment candidate* dibuat dari semua data **training saja**: kandidat dipilih kembali melalui inner CV (mean AP), lalu threshold dihitung dengan inner OOF. Model dilatih pada 1.068 data training dan disimpan dengan `joblib`, siap diintegrasikan ke notebook DSS setelah keputusan metodologis. **Jangan mengganti angka holdout lama dengan evaluasi ulang pada 268 data yang sudah dilihat.**

In [ ]:
cv_final=StratifiedKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=SEED+999)
full_rank=[]
for m,a in CANDIDATES:
    sc=cross_validate(stageA_estimator(m,a),Xa,ya,cv=cv_final,
         scoring={'ap':'average_precision','ba':'balanced_accuracy'},n_jobs=1,error_score='raise')
    full_rank.append((float(sc['test_ap'].mean()),float(sc['test_ba'].mean()),m,a))
final_row=sorted(full_rank,key=lambda x:(-x[0],-x[1],x[2],x[3]))[0]
selected_name,selected_ablation=final_row[2],final_row[3]
final_est=stageA_estimator(selected_name,selected_ablation)
p_oof_final=cross_val_predict(final_est,Xa,ya,cv=cv_final,method='predict_proba',n_jobs=1)[:,1]
final_threshold=choose_threshold(ya,p_oof_final)
final_est.fit(Xa,ya)
joblib.dump(final_est, OUT_DIR/'08_model_A_train_only.joblib',compress=3)
metadata={'model':selected_name,'ablation':selected_ablation,'threshold_train_oof':final_threshold,
          'data_train_count':len(Xa),'data_holdout_count_UNTOUCHED':len(test),
          'positive_proxy_train':int(ya.sum()),'model_score':'P(keanggotaan benchmark Maju+Mandiri sementara)',
          'selection':'3-fold inner CV mean average precision, tie by balanced accuracy',
          'threshold':'optimal F1 on out-of-fold train only',
          'status_label_JADESTA':'BELUM DIVERIFIKASI',
          'holdout_protocol':'Do NOT reuse previously inspected test to claim new unbiased performance',
          'uses_12_numeric_and_tfidf':selected_ablation=='structured_plus_tfidf',
          'feature_columns':TRAIN_INPUTS,
          'seed':SEED,'scikit_learn':sklearn_version}
(OUT_DIR/'08_model_A_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False))
print('Kandidat deployment:',selected_name,'+',selected_ablation,'| Threshold:',final_threshold)
print('Model merupakan STUDI AWAL benchmark proxy, bukan validasi resmi.')

## 13. Ekspor paket hasil + instruksi integrasi DSS

Tabel CSV, grafik, file model, dan manifest diekspor ke satu ZIP. **Status Poin 2 reviewer:** dokumentasi robustness tersedia; keputusan final dan revisi manuskrip harus menunggu audit label resmi dan bila mungkin data evaluasi baru yang belum digunakan.

In [ ]:
report={
 'pipeline':'Stage A robustness on train only',
 'n_total':len(df),'n_train':len(train),'holdout_locked':len(test),
 'stageA_train_positive':int(ya.sum()),'stageA_train_negative':int((ya==0).sum()),
 'screening_models':MODELS_A,'ablations':list(ABLATIONS),
 'outer_folds':OUTER_FOLDS,'outer_repeats':OUTER_REPEATS,'inner_folds':INNER_FOLDS,
 'bootstrap_n':BOOTSTRAP_N,'fast_mode':FAST_MODE,'final_candidate':metadata,
 'limitation':'Proxy labels, missingness shift, previous holdout already inspected',
 'shap':'Tidak diulang: SHAP harus dijalankan setelah keputusan model A final bersama model B.'}
(OUT_DIR/'09_manifest_robustness.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
readme="\n".join([
 "HASIL AUDIT ROBUSTNESS TAHAP A — DSS Desa Wisata",
 "02_screening_12_kombinasi.csv : ablasi x 3 model (eksploratif, train only)",
 "03_nested_folds.csv : pemilihan model dan threshold INNER per outer fold",
 "03_nested_oof_predictions.csv : prediksi outer OOF; tidak menyentuh holdout lama",
 "04_nested_metrics_per_repeat.csv : evaluasi per repeat, bukan pengamatan independen",
 "05_bootstrap_95CI.csv : CI berpasangan tingkat desa atas repeat pertama",
 "06_group_holdout_kabupaten.csv : sensitivitas berdasarkan kabupaten/kota",
 "07_precision_recall_nested.png : kurva OOF train-only",
 "07_frekuensi_pemilihan_kandidat.csv : kestabilan kandidat terpilih",
 "08_model_A_train_only.joblib : calon model A (latih saja 1068 desa)",
 "08_model_A_metadata.json : threshold, fitur, konfigurasi",
 "Label di luar benchmark tidak sama dengan kelas resmi negatif JADESTA.",
 "DILARANG menggunakan ulang holdout yang sebelumnya sudah dilihat untuk mengklaim hasil baru independen.",
 "Hasil model A belum otomatis kompatibel dengan SHAP lama hingga model A+B DSS ditetapkan ulang."]) + "\n"
(OUT_DIR/'README.txt').write_text(readme)
output_zip=OUT_DIR.parent/'HASIL_ROBUSTNESS_TAHAP_A.zip'
with zipfile.ZipFile(output_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.iterdir():
        if p.is_file():z.write(p,p.name)
print('Folder:',OUT_DIR.resolve())
print('ZIP:',output_zip.resolve())
try:
    from google.colab import files
    files.download(str(output_zip))
except ImportError: pass